---
title: Building XML with lxml
---

:::{important} Learning Outcomes
- An understanding of how to structure a python script that generates valid XML
- An application of the EAD structure by realizing it through a process that generates it by using a programmatic approach
- An appreciation of how this may be approached in Python, which could then be connected to a metadata spreadsheet for generating a complete output 
:::

## Why generate XML?

The previous sections demonstrated the processes of parsing XML with basic Python tools; editing, checking, and modifying data for quality control; and validation of XML data stored in a file. This section demonstrates how to build an XML document from scratch.

While it would be unusual to use a program like this to build a document on its own, this illustrates a possible step in a larger workflow. Some possible use cases include:

- Many organiations edit, maintain, and update data in spreadsheets or work on the data in a spreadsheet from a database export. A script like this could be developed to output valid XML documents for input into another system.
- Batch creation and transformation of records in other formats from database exports or cataloging workflows. 
- Creation of samples or stub records, which might feed into a cataloging or processing workflow.
- Converting data from another format into valid syntax for ingest into a local database.
- Producing tests or sample files.
  stub records, converting from another format, or producing test files.

This notebook demonstrates a complete process that demonstrates basic XML generation, illustrated using an EAD3-compliant approach to the tags and attributes. First, the basic structure is established, creating namespaces and a root element.
Then, the script creates the basic EAD structure, including required child elements, attributes, and sample text.
Finally, this complete, and validated, tree is written the tree out to a file. At the end of the process, the scripts write out a minimal EAD3 document for a fictitious collection about Lake Superior called "The Superior Papers."

Rather than the basic ET functions, this section uses the `lxml` library since it provides added functionality with greater control over namespace management and outputs (in this case, including `pretty_print` on output which can help for human readability of files).

In [ ]:
# Setup
from pathlib import Path
from lxml import etree

DATA = Path('..', 'data')
OUT = Path('..', 'output')
OUT.mkdir(exist_ok=True)

# a local copy of the EAD3 XSD, used at the end to check what we generated
EAD3_XSD = DATA / 'ead3.xsd'

# the file we will write
fout = OUT / 'superior-papers-auto-generated.xml'

## Namespaces and a helper function

TODO — bullets to expand:

- Every EAD3 element lives in the EAD3 namespace, so every element we create has to be named with it.
  Writing the full `{http://ead3.archivists.org/schema/}control` each time is error-prone.
- `etree.QName(namespace, tag)` builds the qualified name; a small helper function wraps it.
- `nsmap` on the root element declares the namespaces for the whole document. `{None: ...}` makes the EAD3
  namespace the default (no prefix); a string key such as `'ead'` would produce `ead:control`.
  (Connects back to the prefixed-output behavior seen in the `register_namespace` discussion.)
- Other namespaces (Dublin Core, `xsi`) can be declared the same way if the document needs them.
  TODO: decide whether to keep them in this example.

In [ ]:
EAD3_ns = 'http://ead3.archivists.org/schema/'
XSI_ns = 'http://www.w3.org/2001/XMLSchema-instance'

# None as the key sets the default namespace, so no ead: prefixes in the output
ns = {
    None  : EAD3_ns,
    'xsi' : XSI_ns
}

def ead(tag):
    '''
    Take a string representing an EAD tag name and return it qualified with the EAD3 namespace
    '''
    return etree.QName(EAD3_ns, tag)

def add(parent, tag, text=None, **attrib):
    '''
    Add a child element to parent, optionally with text and attributes, and return it
    '''
    child = etree.SubElement(parent, ead(tag), attrib=attrib)
    if text:
        child.text = text
    return child

TODO — bullets on the helper:

- `add()` combines three steps repeated throughout the process: `SubElement`, setting `.text`, setting attributes.
  Show the unrolled version once (`etree.SubElement(...)` then `.text = ...`) before using the helper.
- `**attrib` collects keyword arguments into a dictionary. Attribute names that are not valid Python
  names (e.g. containing a hyphen or colon) would need the dictionary form, `attrib={...}`.
  TODO: confirm no such attribute is needed in this example.

## Creating the root element and `<control>`

TODO — bullets to expand:

- The root is created with `etree.Element()`, not `SubElement`, because it has no parent.
- `etree.ElementTree(root)` wraps the root so we can write it out with a declaration.
- `<control>` holds information about the finding aid itself (as opposed to the collection it describes):
  record ID, who wrote it, when, and who maintains it.
- Required elements: EAD3 requires `recordid`, `filedesc`, `maintenancestatus`, `maintenanceagency`, and at least
  one of the later declaration or history elements. We found this out by validating (see end of section).
  TODO: decide whether to present the failing version first, as in the validation notebook.

In [ ]:
root = etree.Element(ead('ead'), nsmap=ns)
finding_aid = etree.ElementTree(root)

control = add(root, 'control',
              countryencoding='iso3166-1', dateencoding='iso8601', langencoding='iso639-2b')

# recordid
add(control, 'recordid', '1234', instanceurl='http://example.org/fake-ead.xml')

# filedesc: title and publication statements
filedesc = add(control, 'filedesc')

titlestmt = add(filedesc, 'titlestmt')
add(titlestmt, 'titleproper', 'A Finding Aid for the Superior Papers')

publicationstmt = add(filedesc, 'publicationstmt')
add(publicationstmt, 'publisher', 'University of Michigan School of Information')
add(publicationstmt, 'date', 'September 2022', normal='2022-09-01')

# maintenance information (required)
add(control, 'maintenancestatus', value='new')
maintenanceagency = add(control, 'maintenanceagency')
add(maintenanceagency, 'agencyname', 'University of Michigan School of Information')

maintenancehistory = add(control, 'maintenancehistory')
event = add(maintenancehistory, 'maintenanceevent')
add(event, 'eventtype', value='created')
add(event, 'eventdatetime', '2022-09-01', standarddatetime='2022-09-01')
add(event, 'agenttype', value='human')
add(event, 'agent', 'Jesse Johnston')

## Building `<archdesc>`: the collection description

TODO — bullets to expand:

- `<archdesc>` describes the materials. `level="collection"` states the level of description.
- `<did>` ("descriptive identification") holds the core identifying information: title, repository, and more
  in a full finding aid (dates, extent, language, abstract).
- Text in `<bioghist>` must sit inside a `<p>`; EAD3 does not allow bare text there.
  TODO: this is a good moment to note that the schema, not the library, decides what is allowed. `lxml`
  will happily build an invalid document.
- Comments: `etree.Comment()` inserts an XML comment. Useful for leaving notes in generated files.

In [ ]:
archdesc = add(root, 'archdesc', level='collection')

# did: identifying information about the collection
did = add(archdesc, 'did')
add(did, 'unittitle', 'The Superior Papers')

repository = add(did, 'repository')
corpname = add(repository, 'corpname')
add(corpname, 'part', 'University of Michigan')
add(corpname, 'part', 'School of Information')

# a comment about what is missing
did.append(etree.Comment(' somewhere around here would be additional information like scope and extent '))

# bioghist: note that the text goes in a <p>
bioghist = add(archdesc, 'bioghist')
add(bioghist, 'p', 'This collection is a superior collection of papers for a superior organization and individual.')

## The contents list: nested components

TODO — bullets to expand:

- `<dsc>` (description of subordinate components) holds the contents list.
- Components nest: a series (`<c01>`) can contain files (`<c02>`) that can contain items (`<c03>`).
  Hierarchy is built by passing the parent element to `add()`.
- Each component needs its own `<did>` with a `<unittitle>`; text cannot sit directly in `<c01>`.
- Contrast with a spreadsheet, where this hierarchy is flattened into rows.
  TODO: possibly show a loop that builds components from a list or dictionary (see Challenges).

In [ ]:
dsc = add(archdesc, 'dsc')

# series 1: Business Records
c01 = add(dsc, 'c01', level='series')
add(add(c01, 'did'), 'unittitle', 'Business Records')

# series 2: Correspondence, with a file nested inside
c01 = add(dsc, 'c01', level='series')
add(add(c01, 'did'), 'unittitle', 'Correspondence')

c02 = add(c01, 'c02', level='file')
add(add(c02, 'did'), 'unittitle', 'Personal File')

## Viewing the tree before writing

TODO — bullets to expand:

- `etree.tostring()` returns bytes; `pretty_print=True` adds indentation; `.decode()` makes it printable.
- Check the output before writing a file, in the same way earlier notebooks inspected parsed trees.

In [ ]:
print(etree.tostring(finding_aid, pretty_print=True).decode('utf-8'))

## Adding a schema declaration

TODO — bullets to expand:

- Existing EAD files (e.g. `sample-ead-superior-full.xml`) begin with an `xml-model` processing instruction pointing
  to the schema. Processing instructions are not elements; they sit outside the root and are instructions to software.
- `etree.ProcessingInstruction(target, text)` creates one; `root.addprevious()` places it before the root.
- Reference: [W3C Associating Schemas with XML documents](https://www.w3.org/TR/xml-model/).
- TODO: the original script attempted this with `root.append()`, which places it inside the root and is not valid. `addprevious()` is the fix.

In [ ]:
root.addprevious(etree.ProcessingInstruction(
    'xml-model',
    'href="https://raw.githubusercontent.com/SAA-SDT/EAD3/master/ead3.rng" '
    'type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"'
))

## Writing the file

TODO — bullets to expand:

- `ElementTree.write()` with `xml_declaration=True` and `encoding='utf-8'` adds the `<?xml ...?>` declaration.
- `pretty_print=True` indents the output for human readers. It is only safe because we created the document
  with no meaningful whitespace; applying it to a parsed document with mixed content can alter its text.
  (TODO: link to the prettify discussion in the earlier notebook.)

In [ ]:
finding_aid.write(fout, xml_declaration=True, encoding='utf-8', method='xml', pretty_print=True)
print(f'Wrote {fout}')

## Checking the result

TODO — bullets to expand:

- Generated does not mean valid. Re-parse the file and validate it against the EAD3 schema, as in the previous notebook.
- Validation requires re-parsing so we test what was written, not the in-memory tree.

In [ ]:
schema = etree.XMLSchema(etree.parse(EAD3_XSD))
generated = etree.parse(fout)

if schema.validate(generated):
    print('Valid EAD3')
else:
    print(schema.error_log)

## Challenges

TODO — bullets to expand into exercises:

- Prompt for the title, publisher, and date with `input()` instead of hard-coding them.
- Move each section into a function, e.g. `build_control(root, record_id, instance_url)`, and call them in sequence.
- Build the `<c01>` components from a Python list or a CSV file instead of one by one.
- Remove `maintenanceagency` and validate. Read the error and fix it.
- Add a `<scopecontent>` with a `<p>` to `<archdesc>`; consult the EAD3 tag library for where it belongs.
- Try the same document with a prefixed namespace (`'ead'` as the key) and compare the output.